# 📏 Day 01a: SRP & OCP — Single Responsibility & Open/Closed

---

## 🎯 What You'll Master Today
- SRP: One class, one reason to change
- OCP: Open for extension, closed for modification
- Spot violations and refactor them
- Write SOLID code in interview coding rounds

---

## 🎭 The Analogies

**SRP — Single Responsibility Principle:**
- Think of a **Swiss Army knife** vs **a chef's knife**
- Swiss Army knife does everything poorly. Chef's knife does ONE thing brilliantly.
- Each class should be like a chef's knife — one job, done well.

**OCP — Open/Closed Principle:**
- Think of **electrical outlets**
- You don't rewire the house to plug in a new device. You just plug it in.
- Your code should be **open** for new plugins, **closed** for rewiring.

---

## 📐 SOLID at a Glance

```
╔═══════════════════════════════════════════════════════════════════╗
║  SOLID PRINCIPLES                                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  S — Single Responsibility   One class = one reason to change   ║
║  O — Open/Closed             Open for extension, closed for mod ║
║  L — Liskov Substitution     Subtypes must be substitutable     ║
║  I — Interface Segregation   Many specific > one fat interface  ║
║  D — Dependency Inversion    Depend on abstractions, not concretes║
║                                                                   ║
║  Today: S and O  │  Next: L and I  │  Then: D                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. SRP VIOLATION — The God Class
# ============================================

# ❌ BAD — This class does EVERYTHING
class UserManager:
    """Violates SRP: handles user data, email, and reporting."""
    
    def __init__(self, name, email):
        self.name = name
        self.email = email
    
    def save_to_database(self):
        print(f"Saving {self.name} to DB")  # DB responsibility
    
    def send_welcome_email(self):
        print(f"Sending email to {self.email}")  # Email responsibility
    
    def generate_report(self):
        print(f"Report for {self.name}")  # Reporting responsibility


# What's wrong?
# 3 reasons to change: DB schema changes, email template changes, report format changes
# Any change risks breaking all three features
print("❌ SRP Violation: UserManager has 3 responsibilities")
print("   - Data persistence (save_to_database)")
print("   - Communication (send_welcome_email)")
print("   - Reporting (generate_report)")

In [ ]:
# ============================================
# 2. SRP FIXED — Separate Responsibilities
# ============================================

# ✅ GOOD — Each class has ONE responsibility

class User:
    """Responsibility: Hold user data."""
    def __init__(self, name: str, email: str):
        self.name = name
        self.email = email
    
    def __repr__(self):
        return f"User('{self.name}', '{self.email}')"


class UserRepository:
    """Responsibility: Persist user data."""
    def save(self, user: User):
        print(f"💾 Saving {user.name} to database")


class EmailService:
    """Responsibility: Send emails."""
    def send_welcome(self, user: User):
        print(f"📧 Welcome email sent to {user.email}")


class ReportGenerator:
    """Responsibility: Generate reports."""
    def generate(self, user: User) -> str:
        report = f"📊 Report for {user.name}"
        print(report)
        return report


# Usage — each class does ONE thing
user = User("Sarthak", "sarthak@mail.com")

repo = UserRepository()
repo.save(user)

email = EmailService()
email.send_welcome(user)

reporter = ReportGenerator()
reporter.generate(user)

print("\n✅ Each class has exactly ONE reason to change")

In [ ]:
# ============================================
# 3. OCP VIOLATION — Modifying existing code
# ============================================

# ❌ BAD — must modify this function for every new shape
def calculate_area_bad(shape_type: str, **kwargs) -> float:
    if shape_type == "circle":
        return 3.14159 * kwargs["radius"] ** 2
    elif shape_type == "rectangle":
        return kwargs["width"] * kwargs["height"]
    elif shape_type == "triangle":
        return 0.5 * kwargs["base"] * kwargs["height"]
    # Want to add hexagon? Must MODIFY this function!
    # Every modification risks breaking existing shapes.


print("❌ OCP Violation:")
print(f"   Circle area: {calculate_area_bad('circle', radius=5)}")
print(f"   Rectangle area: {calculate_area_bad('rectangle', width=4, height=6)}")
print("   Adding Pentagon = modifying this function = risk!")

In [ ]:
# ============================================
# 4. OCP FIXED — Open for Extension, Closed for Modification
# ============================================
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstraction — defines the contract."""
    
    @abstractmethod
    def area(self) -> float:
        pass
    
    @abstractmethod
    def __str__(self) -> str:
        pass


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius
    
    def area(self) -> float:
        return math.pi * self.radius ** 2
    
    def __str__(self):
        return f"Circle(r={self.radius})"


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height
    
    def area(self) -> float:
        return self.width * self.height
    
    def __str__(self):
        return f"Rectangle({self.width}x{self.height})"


# ✅ Adding a new shape = NEW class, NO modification of existing code!
class Pentagon(Shape):
    def __init__(self, side: float):
        self.side = side
    
    def area(self) -> float:
        return (math.sqrt(5 * (5 + 2 * math.sqrt(5))) / 4) * self.side ** 2
    
    def __str__(self):
        return f"Pentagon(s={self.side})"


def total_area(shapes: list[Shape]) -> float:
    """This function NEVER needs to change — works with any Shape."""
    return sum(s.area() for s in shapes)


shapes = [Circle(5), Rectangle(4, 6), Pentagon(3)]
for s in shapes:
    print(f"  {s}: area = {s.area():.2f}")
print(f"\n  Total area: {total_area(shapes):.2f}")
print("\n✅ Added Pentagon without modifying Circle, Rectangle, or total_area!")

In [ ]:
# ============================================
# 5. Real-World OCP — Discount System
# ============================================
from abc import ABC, abstractmethod

class DiscountStrategy(ABC):
    @abstractmethod
    def calculate(self, price: float) -> float:
        pass

class NoDiscount(DiscountStrategy):
    def calculate(self, price: float) -> float:
        return price

class PercentageDiscount(DiscountStrategy):
    def __init__(self, percent: float):
        self.percent = percent
    
    def calculate(self, price: float) -> float:
        return price * (1 - self.percent / 100)

class FlatDiscount(DiscountStrategy):
    def __init__(self, amount: float):
        self.amount = amount
    
    def calculate(self, price: float) -> float:
        return max(0, price - self.amount)

# ✅ New discount type — NO modification of existing code!
class BuyOneGetOneFree(DiscountStrategy):
    def calculate(self, price: float) -> float:
        return price / 2


class Order:
    """This class NEVER changes when we add new discounts."""
    def __init__(self, item: str, price: float, discount: DiscountStrategy):
        self.item = item
        self.price = price
        self.discount = discount  # composition — inject the strategy
    
    def final_price(self) -> float:
        return self.discount.calculate(self.price)
    
    def __str__(self):
        return f"{self.item}: ₹{self.price:,.0f} → ₹{self.final_price():,.0f}"


orders = [
    Order("Laptop", 80000, PercentageDiscount(10)),
    Order("Book", 500, FlatDiscount(100)),
    Order("Shirt", 2000, BuyOneGetOneFree()),
    Order("Cable", 300, NoDiscount()),
]

for order in orders:
    print(order)

## 🗺️ SRP & OCP Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  SRP & OCP — QUICK REFERENCE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SRP: One class = one responsibility = one reason to change     ║
║  Smell: God class, class doing data + email + reports           ║
║  Fix: Split into focused classes (User, Repo, EmailService)     ║
║                                                                   ║
║  OCP: Add new behavior WITHOUT modifying existing code          ║
║  Smell: if/elif chains for types, modifying function for new    ║
║         variants                                                 ║
║  Fix: Abstract base class + new subclasses for new behavior     ║
║       Strategy pattern = OCP in action                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is SRP? | A class should have only one reason to change — one responsibility |
| 2 | How to detect SRP violation? | Class name has "And" or "Manager". Methods touch unrelated concerns |
| 3 | What is OCP? | Open for extension (add new behavior), closed for modification (don't touch existing code) |
| 4 | How to implement OCP? | Abstract base class/interface + new subclasses. Strategy/Template patterns |
| 5 | If/elif chain for types — which SOLID violation? | OCP — you must modify existing code to add a new type |
| 6 | God class — which SOLID violation? | SRP — one class doing too many things |
| 7 | How does Strategy pattern relate to OCP? | Strategy = OCP in action. New strategies = new classes, no modification |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • SRP: one class = one job = one reason to change      │
## │  • OCP: extend via new classes, don't modify existing   │
## │  • God class = SRP violation → split it                 │
## │  • if/elif type chains = OCP violation → use polymorphism│
## │  • Strategy pattern = OCP in practice                    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **LSP & ISP** — Liskov Substitution & Interface Segregation